# Baseline Model
### Evidence Retrieval
- Use Doc2Vec to encode test claims and all evidences
- Compute cosine similarity between claims and each evidence
- Select top 5 evidences that have the highest similarity score for each claim

### Claim Classification
- Use Random Forest to predict claim label

In [56]:
import json
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from gensim.models.doc2vec import Doc2Vec
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score


In [31]:
nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
	"""Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
	tokens = word_tokenize(text.lower())
	filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
	return ' '.join(filtered_tokens)
	
# Load JSON data
def load_data(filepath):
	with open(filepath, 'r') as file:
		data = json.load(file)
	return data

[nltk_data] Downloading package punkt to /Users/chenluyao/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [32]:
train_claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')
dev_claims_data = load_data('data/dev-claims.json')
evidence_map = load_data('data/curated/preprocessed_evidence_map.json')
test_claims_data = load_data('data/test-claims-unlabelled.json')

In [33]:
evidence_df = pd.DataFrame(evidence_map.items(), columns=['id', 'evidence'])
evidence_df

,id,evidence
0,evidence-0,john bennet law english entrepreneur agricultu...
1,evidence-1,lindberg began profession career age 16 eventu...
2,evidence-2,boston ladi cambridg vampir weekend
3,evidence-3,gerald franci goyer born octob 20 1936 profess...
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...
...,...,...
1208822,evidence-1208822,also properti contribut garag apart
1208823,evidence-1208823,class fn org fyrd 6110 volda
1208824,evidence-1208824,dragon storm game game collect card game
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...


In [45]:
data_for_dataframe = []
for claim_id, claim_details in train_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids,
            'label': claim_label
        })

# Create DataFrame
train_claims_df = pd.DataFrame(data_for_dataframe)

train_claims_df['evidence_texts'] = train_claims_df['evidence'].apply(
    lambda x: [evidence_map[evidence_id] for evidence_id in x]
)

train_claims_df

,claim,evidence,label,evidence_texts
0,Not only is there no scientific evidence that ...,"[evidence-442946, evidence-1194317, evidence-1...",DISPUTED,[high concentr 100 time atmospher concentr gre...
1,El Niño drove record highs in global temperatu...,"[evidence-338219, evidence-1127398]",REFUTES,[climat chang due natur forc human activ subst...
2,"In 1946, PDO switched to a cool phase.","[evidence-530063, evidence-984887]",SUPPORTS,[evid revers prevail polar mean chang cool sur...
3,Weather Channel co-founder John Coleman provid...,"[evidence-1177431, evidence-782448, evidence-5...",DISPUTED,[convinc scientif evid human releas carbon dio...
4,"""January 2008 capped a 12 month period of glob...","[evidence-1010750, evidence-91661, evidence-72...",NOT_ENOUGH_INFO,"[averag temperatur 47, iranian persian calenda..."
...,...,...,...,...
1223,Climate scientists say that aspects of the cas...,"[evidence-1055682, evidence-1047356, evidence-...",SUPPORTS,[fact climat chang made hurrican harvey deadli...
1224,"In its 5th assessment report in 2013, the IPCC...",[evidence-916755],SUPPORTS,[scientif consensu 2013 updat state ipcc fifth...
1225,"Since the mid 1970s, global temperatures have ...","[evidence-403673, evidence-889933, evidence-11...",NOT_ENOUGH_INFO,"[global warm, multipl independ produc instrume..."
1226,But abnormal temperature spikes in February an...,"[evidence-97375, evidence-562427, evidence-521...",NOT_ENOUGH_INFO,[lower air temperatur record 2010 may influenc...


In [54]:
data_for_dataframe = []
for claim_id, claim_details in dev_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    data_for_dataframe.append({
            'claim': claim_text,
            'evidence': eids,
            'label': claim_label
        })

# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)

dev_claims_df['evidence_texts'] = dev_claims_df['evidence'].apply(
    lambda x: [evidence_map[evidence_id] for evidence_id in x]
)

dev_claims_df

,claim,evidence,label,evidence_texts
0,[South Australia] has the most expensive elect...,"[evidence-67732, evidence-572512]",SUPPORTS,[citat need south australia highest retail pri...
1,when 3 per cent of total annual global emissio...,"[evidence-996421, evidence-1080858, evidence-2...",NOT_ENOUGH_INFO,[2011 unep green economi report state agricult...
2,This means that the world is now 1C warmer tha...,"[evidence-889933, evidence-694262]",SUPPORTS,[multipl independ produc instrument dataset co...
3,"“As it happens, Zika may also be a good model ...","[evidence-422399, evidence-702226, evidence-28...",NOT_ENOUGH_INFO,[genet disord result deleteri mutat due sponta...
4,Greenland has only lost a tiny fraction of its...,"[evidence-52981, evidence-264761, evidence-947...",REFUTES,[iceberg calv happen averag greenland lost 294...
...,...,...,...,...
149,"'To suddenly label CO2 as a ""pollutant"" is a d...","[evidence-409365, evidence-127519, evidence-85...",REFUTES,[state articl 2 convent requir greenhous ga gh...
150,"after a natural orbitally driven warming, atmo...","[evidence-368192, evidence-261690, evidence-20...",NOT_ENOUGH_INFO,[increas atmospher concentr co 2 greenhous gas...
151,Many of the world’s coral reefs are already ba...,"[evidence-1124018, evidence-995813, evidence-1...",NOT_ENOUGH_INFO,[tropic water contain nutrient yet coral reef ...
152,A recent study led by Lawrence Livermore Natio...,[evidence-660755],REFUTES,[2007 studi david douglass cowork conclud 22 c...


In [36]:
model= Doc2Vec.load("d2v.model")

In [37]:
# evidence_df["vector"] = ""
# for i in range(evidence_df.shape[0]):
#     inferred_vector = model.infer_vector(evidence_df["evidence"][i].split())
#     evidence_df["vector"][i] = inferred_vector
# evidence_df.to_csv('data/curated/evidence_vector.csv', index=False)

evidence_df = pd.read_csv('data/curated/evidence_vector.csv', converters={
    'vector': lambda x: np.array(x.strip("[]").split(), dtype='float')})
evidence_df

,id,evidence,vector
0,evidence-0,john bennet law english entrepreneur agricultu...,"[-0.08329561, -0.15579118, -0.17553866, -0.129..."
1,evidence-1,lindberg began profession career age 16 eventu...,"[0.19000088, 0.15242222, -0.45844978, -0.01609..."
2,evidence-2,boston ladi cambridg vampir weekend,"[0.05613353, -0.1325368, 0.15981574, 0.0887365..."
3,evidence-3,gerald franci goyer born octob 20 1936 profess...,"[0.12277374, -0.09053773, -0.3347684, -0.28721..."
4,evidence-4,detect abnorm oxytocinerg function schizoaffec...,"[0.21676677, -0.151133, -0.48791853, 0.1081487..."
...,...,...,...
1208822,evidence-1208822,also properti contribut garag apart,"[0.08558074, -0.04533045, -0.14334618, -0.0497..."
1208823,evidence-1208823,class fn org fyrd 6110 volda,"[0.07639017, -0.07667744, -0.3351644, -0.06447..."
1208824,evidence-1208824,dragon storm game game collect card game,"[0.14750834, -0.05903809, -0.40002766, -0.1349..."
1208825,evidence-1208825,state zeriuani great realm tradit relat tribe ...,"[0.4817398, -0.25093532, -0.49334916, -0.16006..."


In [48]:
data_for_dataframe = []
for claim_id, claim_details in test_claims_data.items():
    claim_text = preprocess_text(claim_details['claim_text'])
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'claim_text_raw': claim_details['claim_text']
        })
    
# Create DataFrame
test_claims_df = pd.DataFrame(data_for_dataframe)
test_claims_df 

,claim_id,claim,claim_text_raw
0,claim-2967,contribut wast heat global climat,The contribution of waste heat to the global c...
1,claim-979,warm weather worsen recent drought includ drie...,“Warm weather worsened the most recent five-ye...
2,claim-1609,greenland lost tini fraction ice mass,Greenland has only lost a tiny fraction of its...
3,claim-1020,global reef crisi necessarili mean extinct cor...,“The global reef crisis does not necessarily m...
4,claim-2599,small amount activ substanc caus larg effect,Small amounts of very active substances can ca...
...,...,...,...
148,claim-293,measur equip get old need replac often requir,When the measuring equipment gets old and need...
149,claim-910,cement iron steel petroleum refin industri cou...,"The cement, iron and steel, and petroleum refi..."
150,claim-2815,new studi surfac warm solar cycl found time hi...,A new peer-reviewed study on Surface Warming a...
151,claim-1652,strong co2 effect observ mani differ measur,The strong CO2 effect has been observed by man...


In [49]:
test_claims_df["vector"] = ""
for i in range(test_claims_df.shape[0]):
    inferred_vector = model.infer_vector(test_claims_df["claim"][i].split())
    test_claims_df["vector"][i] = inferred_vector
test_claims_df.to_csv('data/curated/test_claim_vector.csv', index=False)
test_claims_df 

,claim_id,claim,claim_text_raw,vector
0,claim-2967,contribut wast heat global climat,The contribution of waste heat to the global c...,"[0.028113361, 0.040670097, -0.08705642, 0.0160..."
1,claim-979,warm weather worsen recent drought includ drie...,“Warm weather worsened the most recent five-ye...,"[0.041373424, 0.23307668, -0.130507, -0.148293..."
2,claim-1609,greenland lost tini fraction ice mass,Greenland has only lost a tiny fraction of its...,"[0.09089548, -0.04036503, -0.13701095, -0.0518..."
3,claim-1020,global reef crisi necessarili mean extinct cor...,“The global reef crisis does not necessarily m...,"[0.10290111, -0.17336987, -0.1655826, -0.01663..."
4,claim-2599,small amount activ substanc caus larg effect,Small amounts of very active substances can ca...,"[0.0179828, -0.077818476, -0.25428516, -0.1186..."
...,...,...,...,...
148,claim-293,measur equip get old need replac often requir,When the measuring equipment gets old and need...,"[0.078557484, 0.2160849, -0.20821226, -0.06428..."
149,claim-910,cement iron steel petroleum refin industri cou...,"The cement, iron and steel, and petroleum refi...","[0.23890845, 0.10408038, -0.21086998, -0.18680..."
150,claim-2815,new studi surfac warm solar cycl found time hi...,A new peer-reviewed study on Surface Warming a...,"[0.18100502, -0.07168866, 0.03490305, -0.11313..."
151,claim-1652,strong co2 effect observ mani differ measur,The strong CO2 effect has been observed by man...,"[-0.011019352, 0.075620085, -0.062232275, -0.0..."


In [50]:
X = np.array(test_claims_df['vector'].values.tolist())
y = np.array(evidence_df['vector'].values.tolist())
sim = cosine_similarity(X, y)
print(sim.shape)
    

(153, 1208827)


In [74]:
# get top 5 evidence with highest similarity score with the claim
data = np.zeros((sim.shape[0], 5))
for i in range(sim.shape[0]):
	data[i] = np.argpartition(sim[i], -5)[-5:]
data = data.astype(np.int32)

test_claims_df['top5_evidence_id'] = data.tolist()
test_claims_df = test_claims_df[["claim_id", "claim_text_raw", "top5_evidence_id"]]

# get texts of top 5 evidence
test_claims_df['evidence_texts'] = test_claims_df['top5_evidence_id'].apply(
    lambda x: [evidence_map["evidence-" + str(evidence_id)] for evidence_id in x]
)

test_claims_df.to_csv("data/curated/test_evidence_retrieval.csv", index=False)
test_claims_df

/var/folders/df/4qk5nt6555bggnc39502n5b80000gn/T/ipykernel_58796/1048928725.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_claims_df['evidence_texts'] = test_claims_df['top5_evidence_id'].apply(


,claim_id,claim_text_raw,top5_evidence_id,evidence_texts
0,claim-2967,The contribution of waste heat to the global c...,"[1071666, 103761, 295684, 170137, 449168]",[gari sittler march 14 1952 februari 24 2015 c...
1,claim-979,“Warm weather worsened the most recent five-ye...,"[733944, 947706, 578205, 582624, 655529]","[vote member nation academi record art scienc,..."
2,claim-1609,Greenland has only lost a tiny fraction of its...,"[446869, 1041564, 55042, 1113630, 339133]",[contin larg volum ice present store around 70...
3,claim-1020,“The global reef crisis does not necessarily m...,"[372770, 677084, 916862, 451684, 939312]",[coral casado ortiz born 27 march 1996 spanish...
4,claim-2599,Small amounts of very active substances can ca...,"[816943, 542087, 32162, 941104, 20128]","[javelin throw olymp track field event, amount..."
...,...,...,...,...
148,claim-293,When the measuring equipment gets old and need...,"[1205183, 1016047, 1104945, 435390, 167451]",[usual variou search engin provid keyword sugg...
149,claim-910,"The cement, iron and steel, and petroleum refi...","[1041788, 777489, 277041, 465769, 933046]",[rank 5th 13 gold medal 19 silver medal 25 bro...
150,claim-2815,A new peer-reviewed study on Surface Warming a...,"[968454, 558597, 96353, 917435, 592886]",[origin studi address first reason increas vol...
151,claim-1652,The strong CO2 effect has been observed by man...,"[621985, 112809, 325724, 700583, 211121]",[book tell stori condit would eventu lead diss...


### Claim Classification

In [55]:
# dev_claims_df = pd.read_csv('data/curated/dev_evidence_retrieval.csv', converters={
#     'top5_evidence_id': lambda x: np.array(x.strip("[]").split(','), dtype='int')})
# dev_claims_df

In [59]:
# combine claim text and evidence texts
X_train = train_claims_df['claim'] + train_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))
y_train = train_claims_df['label']

X_dev = dev_claims_df['claim'] + dev_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))
y_dev = dev_claims_df['label']

X_test = test_claims_df['claim_text_raw'] + test_claims_df['evidence_texts'].apply(lambda x: ' '.join(x))

count_vectorizer = CountVectorizer()
X_train_count = count_vectorizer.fit_transform(X_train)
X_dev_count = count_vectorizer.transform(X_dev)
X_test_count = count_vectorizer.transform(X_test)

In [57]:
# Hyperparameters
n_estimators_values = [50, 100, 200]
max_depth_values = [None, 10, 20]

accuracy_scores_rf = []
for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:
        rf_classifier = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        rf_classifier.fit(X_train_count, y_train)
        y_pred_rf = rf_classifier.predict(X_dev_count)
        
        accuracy_rf = accuracy_score(y_dev, y_pred_rf)
        accuracy_scores_rf.append(((n_estimators, max_depth), accuracy_rf))
        print(f"n_estimators = {n_estimators}, max_depth = {max_depth}: Accuracy = {accuracy_rf}")

print("Accuracy scores for Random Forest:")
for params, accuracy in accuracy_scores_rf:
    print(f"Parameters: {params}, Accuracy: {accuracy}")

n_estimators = 50, max_depth = None: Accuracy = 0.551948051948052
n_estimators = 50, max_depth = 10: Accuracy = 0.5
n_estimators = 50, max_depth = 20: Accuracy = 0.5324675324675324
n_estimators = 100, max_depth = None: Accuracy = 0.538961038961039
n_estimators = 100, max_depth = 10: Accuracy = 0.5064935064935064
n_estimators = 100, max_depth = 20: Accuracy = 0.4935064935064935
n_estimators = 200, max_depth = None: Accuracy = 0.538961038961039
n_estimators = 200, max_depth = 10: Accuracy = 0.512987012987013
n_estimators = 200, max_depth = 20: Accuracy = 0.525974025974026
Accuracy scores for Random Forest:
Parameters: (50, None), Accuracy: 0.551948051948052
Parameters: (50, 10), Accuracy: 0.5
Parameters: (50, 20), Accuracy: 0.5324675324675324
Parameters: (100, None), Accuracy: 0.538961038961039
Parameters: (100, 10), Accuracy: 0.5064935064935064
Parameters: (100, 20), Accuracy: 0.4935064935064935
Parameters: (200, None), Accuracy: 0.538961038961039
Parameters: (200, 10), Accuracy: 0.5129

In [75]:
# Apply to Test Set
rf_classifier = RandomForestClassifier(n_estimators=50, max_depth=None, random_state=42)
rf_classifier.fit(X_train_count, y_train)
y_pred = rf_classifier.predict(X_test_count)
test_claims_df["label"] = y_pred
test_claims_df['evidences'] = test_claims_df['top5_evidence_id'].apply(
    lambda x: ["evidence-" + str(evidence_id) for evidence_id in x]
)
test_claims_df.drop(columns=['evidence_texts', 'top5_evidence_id'], inplace=True)
test_claims_df.rename(columns={"claim_text_raw": "claim_text", "label": "claim_label"}, inplace=True)
test_claims_df.set_index('claim_id', inplace=True)
test_claims_df

,claim_text,claim_label,evidences
claim_id,,,
claim-2967,The contribution of waste heat to the global c...,SUPPORTS,"[evidence-1071666, evidence-103761, evidence-2..."
claim-979,“Warm weather worsened the most recent five-ye...,SUPPORTS,"[evidence-733944, evidence-947706, evidence-57..."
claim-1609,Greenland has only lost a tiny fraction of its...,SUPPORTS,"[evidence-446869, evidence-1041564, evidence-5..."
claim-1020,“The global reef crisis does not necessarily m...,SUPPORTS,"[evidence-372770, evidence-677084, evidence-91..."
claim-2599,Small amounts of very active substances can ca...,SUPPORTS,"[evidence-816943, evidence-542087, evidence-32..."
...,...,...,...
claim-293,When the measuring equipment gets old and need...,SUPPORTS,"[evidence-1205183, evidence-1016047, evidence-..."
claim-910,"The cement, iron and steel, and petroleum refi...",SUPPORTS,"[evidence-1041788, evidence-777489, evidence-2..."
claim-2815,A new peer-reviewed study on Surface Warming a...,SUPPORTS,"[evidence-968454, evidence-558597, evidence-96..."


In [77]:
# convert to json file
from json import loads
result = test_claims_df.to_json(orient="index")
with open('data/curated/test-output.json', 'w') as f:
    f.write(result)